# 第 9 集 — 批量渲染:Max 里很难做的事

⏱ 约 25 分钟 · 难度 ★★☆

**这一集控制的音乐元素:没有新的。** 这一集的主题是 **数量**。

**Max 里的对应:** 想象你要在 Max 里随机拧 14 个旋钮、录 1 秒、存成文件、记下旋钮的值,然后重复 10000 次。可以做,但很别扭。在 Python 里是十几行。

**你会听到:** 20 个由电脑随机 "拧" 出来的声音。有的好听,有的奇怪,有的几乎没声音 — 这是正常的。

**学完你能做到:**
- 用随机数发生器,并让结果可以重现
- 解释为什么 cutoff 和时间类参数要 "按倍数" 随机
- 批量渲染、存 wav、存参数
- 把存下来的东西读回来,验证 "声音 ↔ 参数" 是配对的
- 把一批声音整理成两张表

**为什么这很重要:** 每个声音都附带一份 "它是用哪些参数做出来的" 记录。这种 **声音 ↔ 参数** 的配对,正是训练 machine learning 模型(比如 "听一个声音,猜出 synth 参数")需要的数据。

## 工具箱

前几集做好的东西,原样搬过来。运行一下就行,不用重读(难的地方都有 `#` 注释)。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip 把超过 ±1 的部分压住,防止爆音
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # 只画开头这么多个 sample
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

import os
import json
import time
from scipy.io import wavfile

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # 表里每个位置的 phase,0 到 1
    table = np.zeros(N)                        # 先做一张全是 0 的空表
    for k, amp in enumerate(harmonic_amps, start=1):
        # 第 k 个 harmonic:在表里走 k 个周期,音量 amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize:峰值变成 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # 第 f 张表有几个 harmonic:从 1 个均匀增加到 max_harmonics 个
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # 叠成二维 array:F 行 × N 列

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- 在哪两张表之间 ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # 0..1 换算成 0..F-1 的 "表编号",可以带小数
    f0 = np.floor(fpos).astype(int)            # 下面那张表
    f1 = np.minimum(f0 + 1, F - 1)             # 上面那张表(不超过最后一张)
    ffrac = fpos - f0                          # 离下面那张有多远,0..1

    # --- 在表里的哪两个位置之间(和第 3 集一样)---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # 左边的位置
    i1 = (i0 + 1) % N                          # 右边的位置,到表尾绕回 0
    frac = p - np.floor(p)                     # 小数部分

    # 先在每张表内部插值,再在两张表之间插值
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # 时间不能是 0,否则下面会除以 0
    # 按住期间:(0,0) → (a,1) → (a+d,s),之后停在 s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # 松开那一刻 envelope 有多高(可能还没走完 attack)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # 从那个高度用 r 秒直线降到 0;np.clip 让它降到 0 后不再变负
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # 松开前用 held,松开后用 released

def lowpass(x, cutoff, res=0.0):
    x = np.asarray(x, dtype=float)
    n_samples = len(x)
    # cutoff 给一个数或一个 array 都行:统一变成和声音一样长的 array
    cutoff = np.clip(np.broadcast_to(cutoff, n_samples), 20, sr * 0.45)

    # --- 把 cutoff / res 换算成 filter 内部用的系数(每个 sample 一组)---
    g = np.tan(np.pi * cutoff / sr)
    k = 1.414 * (1 - res) + 0.05 * res         # damping:res 越大 k 越小,共鸣越强
    a1 = 1 / (1 + g * (g + k))
    a2 = g * a1
    a3 = g * a2

    # .tolist() 把 array 变成普通 list:在循环里逐个取数时快很多
    xs, A1, A2, A3 = x.tolist(), a1.tolist(), a2.tolist(), a3.tolist()
    y = [0.0] * n_samples
    ic1 = ic2 = 0.0                            # 两个状态变量 = gen~ 里的两个 history

    for n in range(n_samples):                 # 一个 sample 一个 sample 按顺序算
        v3 = xs[n] - ic2
        v1 = A1[n] * ic1 + A2[n] * v3
        v2 = ic2 + A2[n] * ic1 + A3[n] * v3    # v2 就是 lowpass 的输出
        ic1 = 2 * v1 - ic1                     # 更新状态,留给下一个 sample 用
        ic2 = 2 * v2 - ic2
        y[n] = v2
    return np.array(y)

def phasor(freq, n_samples):
    # freq / sr = 这一个 sample 里 phase 往前走多少
    # cumsum 把每一步累加起来;% 1 只留小数部分
    return np.cumsum(np.broadcast_to(freq, n_samples) / sr) % 1

def mtof(m):
    return 440 * 2 ** ((m - 69) / 12)          # MIDI 69 = 440 Hz;每 12 个半音翻一倍

def synth(p, dur=2.0, gate=1.2):
    n = int(sr * dur)
    t = np.arange(n) / sr

    # 两个 envelope
    amp_env = adsr(t, p["amp_a"], p["amp_d"], p["amp_s"], p["amp_r"], gate)
    mod_env = adsr(t, p["mod_a"], p["mod_d"], p["mod_s"], p["mod_r"], gate)

    # oscillator:wt_pos 旋钮 + mod env 推上去的量
    phase = phasor(mtof(p["note"]), n)
    x = osc_wavetable(frames, phase, p["wt_pos"] + p["wt_env"] * mod_env)

    # filter:cutoff 旋钮,被 mod env 往上推 cut_env 个八度
    x = lowpass(x, p["cutoff"] * 2 ** (p["cut_env"] * mod_env), p["res"])

    return x * amp_env                         # VCA

frames = make_frames()

## 1. 随机数

`rng` 是一个随机数发生器(random number generator)。Max 里对应 `random` 和 `noise~`。

- `rng.uniform(a, b)`:在 a 和 b 之间随机取一个数,每个位置的机会均等
- `rng.integers(a, b)`:随机取一个整数,从 a 到 b - 1

每运行一次,得到的数都不一样:

In [ ]:
rng = np.random.default_rng()

print(rng.uniform(0, 1))
print(rng.uniform(0, 1))
print(rng.integers(28, 61))

### Seed:可以重现的随机

做实验时,"每次都不一样" 是个麻烦:你没法回头再生成同一批声音。

给发生器一个 **seed**(任意一个整数),它就会每次产生 **同一串** 随机数。换一个 seed,就换一串。

下面两个发生器 seed 都是 0,所以输出完全相同。第三个 seed 是 1,输出不同。

In [ ]:
first = np.random.default_rng(seed=0)
second = np.random.default_rng(seed=0)
other = np.random.default_rng(seed=1)

print("seed 0:", np.round(first.uniform(0, 1, 4), 3))
print("seed 0:", np.round(second.uniform(0, 1, 4), 3))
print("seed 1:", np.round(other.uniform(0, 1, 4), 3))

## 2. 怎么随机一个 cutoff

cutoff 的范围是 80 到 8000 Hz。直接 `rng.uniform(80, 8000)` 行不行?

第 7 集的结论:耳朵按 **倍数** 听频率。80 到 8000 大约是 6.6 个八度。但如果在 Hz 上均匀取:

- 4000–8000 Hz(最高的 **一个** 八度)占了整个范围的一半
- 80–160 Hz(最低的一个八度)只占 1%

结果就是:随机出来的声音大多数都很亮,几乎没有闷的。

解决:在 **log 上** 均匀取,再换算回来。这叫 log-uniform,和 `geomspace` 是同一个想法。

- `np.log(lo)`、`np.log(hi)`:把两端换成 log
- 在 log 值之间均匀随机
- `np.exp(...)`:换回原来的单位

下面各取 2000 个,画成直方图(每个柱子 = 落在那个区间里的个数)。横轴是按八度画的。

**应该看到:** 左图(普通 uniform)几乎全挤在右边;右图(log-uniform)各个八度大致一样多。

In [ ]:
rng = np.random.default_rng(seed=0)

def log_uniform(lo, hi):
    # 在 log 值之间均匀取,再用 exp 换回来
    return float(np.exp(rng.uniform(np.log(lo), np.log(hi))))

plain = []
logged = []
for i in range(2000):
    plain.append(rng.uniform(80, 8000))
    logged.append(log_uniform(80, 8000))

bins = np.geomspace(80, 8000, 30)              # 按倍数划分的 30 个区间
fig, axes = plt.subplots(1, 2, figsize=(11, 3), sharey=True)
axes[0].hist(plain, bins=bins)
axes[0].set_title("uniform(80, 8000)")
axes[1].hist(logged, bins=bins)
axes[1].set_title("log_uniform(80, 8000)")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("cutoff (Hz)")
plt.show()

**规则:**

- 频率、时间类参数(`cutoff`、attack、decay、release)→ `log_uniform`
- 本来就是 0 到 1 的参数(`wt_pos`、`res`、sustain、amount)→ 普通 `uniform`

## 3. 随机一整组参数

14 个参数,每个按上面的规则取。`float(...)` 和 `int(...)` 是把 numpy 的数变成普通 Python 数字,后面存文件时需要。

为了每次运行这个 cell 都得到同样的结果,这里重新建了一次 `rng`。

In [ ]:
rng = np.random.default_rng(seed=0)

def random_params():
    return dict(
        note=int(rng.integers(28, 61)),            # MIDI 28..60
        wt_pos=float(rng.uniform(0, 1)),
        wt_env=float(rng.uniform(0, 1)),
        cutoff=log_uniform(80, 8000),              # 频率 → 按倍数
        res=float(rng.uniform(0, 0.9)),
        cut_env=float(rng.uniform(0, 5)),
        amp_a=log_uniform(0.002, 0.5),             # 时间 → 按倍数
        amp_d=log_uniform(0.02, 0.5),
        amp_s=float(rng.uniform(0, 1)),
        amp_r=log_uniform(0.02, 0.4),
        mod_a=log_uniform(0.002, 0.5),
        mod_d=log_uniform(0.02, 0.5),
        mod_s=float(rng.uniform(0, 1)),
        mod_r=log_uniform(0.02, 0.4),
    )

p = random_params()
for name in p:
    print(f"{name:8} = {p[name]:.3f}")

`f"{name:8} = {p[name]:.3f}"` 是 **f-string**:引号前面加 `f`,花括号里的东西会被换成它的值。`:8` 是 "占 8 格宽"(为了对齐),`:.3f` 是 "保留 3 位小数"。

听一下这一组。时长 1 秒,按住 0.6 秒。

**应该听到:** 一个随机的声音 — 不一定好听。

In [ ]:
play(synth(p, dur=1.0, gate=0.6))

## 4. 批量渲染

每一轮:

1. 随机一组参数
2. `synth()` 渲染 1 秒
3. 存成 `.wav`
4. 把参数和声音各记进一个 list

最后把所有参数存成一个 `params.json`。

- `os.makedirs("dataset", exist_ok=True)`:在 notebook 旁边建一个 `dataset` 文件夹(已经有就跳过)
- `f"{i:03d}.wav"`:把数字 `i` 变成三位数的文件名 — `000.wav`、`001.wav` ……
- `os.path.join(a, b)`:把文件夹名和文件名拼成路径
- **JSON** 是存 `dict` 和 list 的标准文本格式;`json.dump` 写,`json.load` 读

In [ ]:
rng = np.random.default_rng(seed=0)            # seed 固定:每次运行得到同一批 20 个声音

n_sounds = 20
out_dir = "dataset"
os.makedirs(out_dir, exist_ok=True)

all_params = []
sounds = []
start = time.time()

for i in range(n_sounds):
    p = random_params()                                        # 1. 随机拧旋钮
    x = synth(p, dur=1.0, gate=0.6)                            # 2. 渲染
    audio = np.clip(0.3 * x, -1, 1).astype(np.float32)         # 降音量、防爆、转成 wav 用的格式
    wavfile.write(os.path.join(out_dir, f"{i:03d}.wav"), sr, audio)     # 3. 存盘
    all_params.append(p)                                       # 4. 记下参数
    sounds.append(x)

with open(os.path.join(out_dir, "params.json"), "w") as f:
    json.dump(all_params, f, indent=2)

elapsed = time.time() - start
print(f"rendered {n_sounds} sounds in {elapsed:.1f} s")
print(f"at this speed, 10000 sounds would take about {10000 * elapsed / n_sounds / 60:.0f} minutes")

`with open(...) as f:` 是 "打开文件 → 做下面缩进的事 → 自动关上"。

去 notebook 所在的文件夹看一下:`dataset/` 里应该有 `000.wav` 到 `019.wav` 和一个 `params.json`。用文本编辑器打开 `params.json` 就能看到每个声音的参数。

## 5. 听和看

### 前 6 个

**应该听到:** 6 个彼此很不一样的短音 — 音高、明暗、长短都不同。每个上面印着它的几个关键参数,试着把听到的和数字对上:`note` 大的音高,`cutoff` 大的亮,`amp_a` 大的淡入慢。

In [ ]:
for i in range(6):
    p = all_params[i]
    print(f"{i:03d}  note={p['note']}  cutoff={p['cutoff']:.0f} Hz  res={p['res']:.2f}  "
          f"cut_env={p['cut_env']:.1f} oct  amp_a={p['amp_a']:.3f} s")
    display(play(sounds[i]))

### 20 个一眼扫过

每一格是一个声音的 spectrogram(横轴时间,纵轴频率,越亮越响)。

**应该看到:** 20 张很不一样的图 — 有的只有底下几条线(闷),有的铺满(亮),有的只有开头一小块(短),有的有明显向下收的形状(filter envelope)。

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(12, 7), sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    with np.errstate(divide="ignore"):         # 完全静音的地方算 dB 会警告,这里忽略它
        ax.specgram(sounds[i] + 1e-9, NFFT=1024, Fs=sr, noverlap=768, cmap="magma", vmin=-120)
    ax.set_ylim(0, 6000)
    ax.set_title(f"{i:03d}", fontsize=8)
fig.supxlabel("time (s)")
fig.supylabel("frequency (Hz)")
plt.tight_layout()
plt.show()

## 6. 读回来,验证配对

"声音 ↔ 参数是配对的" 是什么意思?意思是:**拿存下来的参数重新渲染,应该得到和存下来的 wav 一模一样的声音。**

验证一下第 3 号:

1. 从 `params.json` 读出第 3 组参数
2. 用它重新 `synth()`
3. 从 `003.wav` 读出声音
4. 比较两者

`np.allclose(a, b, atol=1e-6)`:两个 array 是否 "几乎相等"(每个数相差不超过百万分之一)。

In [ ]:
with open(os.path.join(out_dir, "params.json")) as f:
    loaded_params = json.load(f)

p3 = loaded_params[3]
rerendered = np.clip(0.3 * synth(p3, dur=1.0, gate=0.6), -1, 1)       # 和存盘时一样的处理

file_sr, from_file = wavfile.read(os.path.join(out_dir, "003.wav"))

print("sample rate in file :", file_sr)
print("same length         :", len(rerendered) == len(from_file))
print("same sound          :", np.allclose(rerendered, from_file, atol=1e-6))

`same sound: True` 说明:参数完整地描述了这个声音。这就是这个数据集的价值所在。

## 7. 质量检查

随机出来的声音响度差别很大,有些会非常小声(比如 cutoff 很低、sustain 又很低)。做数据集时通常要把它们找出来。

**RMS** 是 "平均响度" 的一种算法:每个 sample 平方 → 取平均 → 开根号。

**应该看到:** 20 根柱子,高矮不一。红线以下的算 "太小声"。

In [ ]:
rms = []
for x in sounds:
    rms.append(np.sqrt(np.mean(x ** 2)))       # 平方 → 平均 → 开根号
rms = np.array(rms)

threshold = 0.1

plt.figure(figsize=(9, 3))
plt.bar(np.arange(n_sounds), rms)
plt.axhline(threshold, color="red")
plt.xlabel("sound number")
plt.ylabel("RMS")
plt.xticks(np.arange(n_sounds))
plt.show()

quiet = np.where(rms < threshold)[0]           # RMS 低于阈值的那些编号
print("too quiet:", quiet)

听一下列出来的第一个,确认它确实比其他的小声,并看看它的参数,想想为什么。

In [ ]:
if len(quiet) > 0:
    i = int(quiet[0])
    print("sound", i, " note =", all_params[i]["note"], " cutoff =", round(all_params[i]["cutoff"]),
          "Hz  cut_env =", round(all_params[i]["cut_env"], 1), " amp_s =", round(all_params[i]["amp_s"], 2))
    display(play(sounds[i]))
else:
    print("no sound is below the threshold")

## 8. 整理成两张表

machine learning 模型不认 `dict` 和文件夹,它认 **表**(二维 array):每一行是一个例子。

- **X**:声音。20 行 × 44100 列(每行是一个 1 秒的声音)
- **Y**:参数。20 行 × 14 列(每行是那个声音的 14 个参数)

**第 i 行的 X 和第 i 行的 Y 是一对。** "听声音猜参数" 的模型,就是学习从 X 的一行算出 Y 的同一行。

In [ ]:
names = list(all_params[0].keys())             # 14 个参数名,顺序固定

rows = []
for p in all_params:
    row = []
    for name in names:
        row.append(p[name])                    # 按同样的顺序取出 14 个值
    rows.append(row)

X = np.array(sounds)
Y = np.array(rows)

print("names:", names)
print("X shape:", X.shape, "  (sounds x samples)")
print("Y shape:", Y.shape, "     (sounds x parameters)")
print()
print("row 3 of Y:", np.round(Y[3], 3))

到这里,一个(很小的)数据集就完整了。把 `n_sounds` 改成 10000,同样的代码就能产出一个真正可以用来训练的数据集。

## 常见错误

| 现象 | 原因 |
|---|---|
| 每次运行得到的声音都不一样 | 建 `rng` 时没有给 seed,或者没有重新运行建 `rng` 的那一行 |
| `Object of type float32 is not JSON serializable` | 参数里混进了 numpy 的数。用 `float(...)` / `int(...)` 包一下 |
| 读回来 `same sound: False` | 重新渲染时的 `dur`、`gate` 或音量处理和存盘时不一样 |
| wav 文件在别的软件里爆音 | 存盘前没有 `np.clip` 或没有降音量 |

## 练习

**1.** 把 seed 改成 1,重新运行第 4 节和第 5 节。**应该听到:** 完全不同的 20 个声音。再改回 0,应该回到原来那一批。

**2.** 做一个 "只有 bass" 的数据集:`note` 限制在 28–40,`cutoff` 限制在 80–1000。提示:复制 `random_params`,改两行。

<details><summary>答案</summary>

```python
def random_bass():
    p = random_params()
    p["note"] = int(rng.integers(28, 41))
    p["cutoff"] = log_uniform(80, 1000)
    return p

rng = np.random.default_rng(seed=5)
for i in range(4):
    display(play(synth(random_bass(), dur=1.0, gate=0.6)))
```
</details>

**3.** 在 `Y` 里找出 cutoff 最高的那个声音并播放。提示:`names.index("cutoff")` 给出 cutoff 是第几列;`np.argmax(一列)` 给出最大值在第几行。

<details><summary>答案</summary>

```python
col = names.index("cutoff")
i = int(np.argmax(Y[:, col]))          # Y[:, col] = 所有行的第 col 列
print("sound", i, "cutoff", round(Y[i, col]))
play(sounds[i])
```
</details>

## 小结

| | Max / RNBO | Python |
|---|---|---|
| 实时演奏、拧旋钮 | 很擅长 | 不擅长 |
| 自动生成一万个声音并记录参数 | 别扭 | 一个循环 |

| Max | Python |
|---|---|
| `random` | `rng.uniform(a, b)`、`rng.integers(a, b)` |
| `seed` message | `np.random.default_rng(seed=0)` |
| `sfrecord~` | `wavfile.write(...)` |
| `dict` 的 `write` | `json.dump(...)` |

两者不是替代关系,而是分工:

- **Python**:离线、批量、可重现 — 做数据、做分析、做实验
- **Max / RNBO**:实时、可交互 — 做乐器、做给人用的东西

**下一集(最后一集):** 用 Python 看声音,然后把做好的 wavetable 和 preset 送回 Max。